# Statistical Analysis & Equity Assessment

**Author: Rohan Raval**

**Objectives:**
1. Correlation analysis
2. Regression modeling: predictors of service access
3. Equity analysis by income/race
4. Validation and sensitivity analysis

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, classification_report
import statsmodels.api as sm

sns.set_style("whitegrid")
pd.set_option('display.max_columns', 120)

candidates = [Path("."), Path(".."), Path("../..")]
for cand in candidates:
    if (cand / "data" / "processed").exists():
        PROJECT_ROOT = cand.resolve()
        break

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_ACS = DATA_DIR / "processed" / "acs5" / "2023"

print("PROJECT_ROOT:", PROJECT_ROOT)

## Load Data

In [ ]:
metrics = pd.read_csv(PROCESSED_ACS / "tract_opportunity_desert_metrics_2023.csv", dtype={"GEOID": str})

metrics_clean = metrics.dropna(subset=[
    "youth_10_19_per_1k", "services_per_1k_youth_10_19",
    "median_hh_income_2023usd", "zero_veh_share"
]).copy()

print(f"Tracts: {len(metrics_clean)}")
print(f"\nColumns: {metrics_clean.columns.tolist()}")

## 1. Correlation Analysis

In [ ]:
corr_vars = [
    "youth_10_19_per_1k",
    "services_per_1k_youth_10_19",
    "median_hh_income_2023usd",
    "zero_veh_share",
    "stops_within_500m_per_tract",
    "avg_distance_to_nearest_service_km"
]

corr_data = metrics_clean[corr_vars].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation Matrix: Key Variables")
plt.tight_layout()
plt.show()

print("\nKey correlations with service access:")
print(corr_matrix["services_per_1k_youth_10_19"].sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0,0].scatter(metrics_clean["median_hh_income_2023usd"], 
                   metrics_clean["services_per_1k_youth_10_19"], alpha=0.5)
axes[0,0].set_xlabel("Median HH Income ($)")
axes[0,0].set_ylabel("Services per 1k Youth")

axes[0,1].scatter(metrics_clean["zero_veh_share"],
                   metrics_clean["services_per_1k_youth_10_19"], alpha=0.5)
axes[0,1].set_xlabel("Zero Vehicle Share")
axes[0,1].set_ylabel("Services per 1k Youth")

axes[1,0].scatter(metrics_clean["youth_10_19_per_1k"],
                   metrics_clean["services_per_1k_youth_10_19"], alpha=0.5)
axes[1,0].set_xlabel("Youth Density (per 1k)")
axes[1,0].set_ylabel("Services per 1k Youth")

axes[1,1].scatter(metrics_clean["stops_within_500m_per_tract"],
                   metrics_clean["services_per_1k_youth_10_19"], alpha=0.5)
axes[1,1].set_xlabel("Transit Stops (500m)")
axes[1,1].set_ylabel("Services per 1k Youth")

plt.tight_layout()
plt.show()

## 2. Regression Analysis

In [ ]:
reg_data = metrics_clean[[
    "services_per_1k_youth_10_19",
    "youth_10_19_per_1k",
    "median_hh_income_2023usd",
    "zero_veh_share",
    "stops_within_500m_per_tract"
]].dropna()

y = reg_data["services_per_1k_youth_10_19"]
X = reg_data[[
    "youth_10_19_per_1k",
    "median_hh_income_2023usd",
    "zero_veh_share",
    "stops_within_500m_per_tract"
]]

X_with_const = sm.add_constant(X)
model = sm.OLS(y, X_with_const).fit()

print(model.summary())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(model.fittedvalues, model.resid, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel("Fitted Values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted")

sm.qqplot(model.resid, line='45', ax=axes[1])
axes[1].set_title("Q-Q Plot")

plt.tight_layout()
plt.show()

In [ ]:
desert_data = metrics_clean[[
    "desert_flag",
    "youth_10_19_per_1k",
    "median_hh_income_2023usd",
    "zero_veh_share",
    "stops_within_500m_per_tract"
]].dropna()

y_desert = desert_data["desert_flag"]
X_desert = desert_data[[
    "youth_10_19_per_1k",
    "median_hh_income_2023usd",
    "zero_veh_share",
    "stops_within_500m_per_tract"
]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_desert)

log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_scaled, y_desert)

y_pred = log_model.predict(X_scaled)

print("Logistic Regression: Predicting Desert Status")
print("\nCoefficients:")
for name, coef in zip(X_desert.columns, log_model.coef_[0]):
    print(f"  {name}: {coef:.4f}")

print("\nClassification Report:")
print(classification_report(y_desert, y_pred, target_names=["Non-Desert", "Desert"]))

## 3. Equity Analysis

In [ ]:
metrics_clean["income_quintile"] = pd.qcut(
    metrics_clean["median_hh_income_2023usd"],
    q=5,
    labels=["Lowest", "Low", "Middle", "High", "Highest"]
)

equity_summary = metrics_clean.groupby("income_quintile").agg({
    "services_per_1k_youth_10_19": ["mean", "median", "std"],
    "avg_distance_to_nearest_service_km": ["mean", "median"],
    "stops_within_500m_per_tract": ["mean", "median"],
    "desert_flag": "sum",
    "GEOID": "count"
})

equity_summary.columns = ['_'.join(col).strip() for col in equity_summary.columns.values]
equity_summary = equity_summary.rename(columns={"GEOID_count": "n_tracts"})

print("Service Access by Income Quintile:")
print(equity_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_clean.boxplot(
    column="services_per_1k_youth_10_19",
    by="income_quintile",
    ax=axes[0]
)
axes[0].set_title("Service Access by Income Quintile")
axes[0].set_xlabel("Income Quintile")
axes[0].set_ylabel("Services per 1k Youth")

desert_by_income = metrics_clean.groupby("income_quintile")["desert_flag"].sum()
desert_by_income.plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Desert Tracts by Income Quintile")
axes[1].set_xlabel("Income Quintile")
axes[1].set_ylabel("Number of Desert Tracts")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
def gini_coefficient(x):
    n = len(x)
    x_sorted = np.sort(x)
    cumsum = np.cumsum(x_sorted)
    return (2 * np.sum((np.arange(1, n+1)) * x_sorted)) / (n * cumsum[-1]) - (n + 1) / n

services_values = metrics_clean["services_per_1k_youth_10_19"].dropna()
gini = gini_coefficient(services_values)

print(f"Gini Coefficient for Service Access: {gini:.4f}")
print("(0 = perfect equality, 1 = perfect inequality)")

## 4. Sensitivity Analysis

In [ ]:
thresholds = [0.20, 0.25, 0.30]

results = []
for thresh in thresholds:
    p_youth = metrics_clean["youth_10_19_per_1k"].quantile(0.75)
    p_services = metrics_clean["services_per_1k_youth_10_19"].quantile(thresh)
    p_zero_veh = metrics_clean["zero_veh_share"].quantile(0.75)
    
    desert_alt = (
        (metrics_clean["youth_10_19_per_1k"] >= p_youth) &
        (metrics_clean["services_per_1k_youth_10_19"] <= p_services) &
        (metrics_clean["zero_veh_share"] >= p_zero_veh)
    )
    
    n_deserts = desert_alt.sum()
    pct = 100 * n_deserts / len(metrics_clean)
    
    results.append({
        "services_percentile": thresh,
        "n_deserts": n_deserts,
        "pct_tracts": pct
    })

sensitivity_df = pd.DataFrame(results)
print("Sensitivity to Service Threshold:")
print(sensitivity_df)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sensitivity_df["services_percentile"], sensitivity_df["n_deserts"], marker='o')
ax.set_xlabel("Service Threshold (Percentile)")
ax.set_ylabel("Number of Desert Tracts")
ax.set_title("Sensitivity to Service Access Threshold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Save Analysis Results

In [ ]:
metrics_clean.to_csv(PROCESSED_ACS / "tract_with_equity_analysis.csv", index=False)
print(f"Saved: {PROCESSED_ACS / 'tract_with_equity_analysis.csv'}")

with open(PROCESSED_ACS / "regression_summary.txt", "w") as f:
    f.write(str(model.summary()))
print(f"Saved: {PROCESSED_ACS / 'regression_summary.txt'}")